<a href="https://colab.research.google.com/github/FishyFoshy/COMP3608-Project/blob/main/Dataset_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Cleaning of Dataset 3

In [ ]:
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)

# Load datasets
df3 = pd.read_csv('Dataset 3.csv')

# Drops all unnecessary columns
df3 = df3.drop(columns=['id', 'date', 'yr_renovated', 'zipcode', 'sqft_living15', 'sqft_lot15'])

# Drops all with null values
df3.dropna(inplace=True)

# Separates non numerical columns from numerical ones
non_numerical_columns = ['price']
numerical_features = [col for col in df3.columns if col not in non_numerical_columns]

# Scale numerical features
scaler = StandardScaler()
df3[numerical_features] = scaler.fit_transform(df3[numerical_features])

# One-Hot Encodes all categorical data
categorical_cols_to_encode = df3.select_dtypes(include='object').columns
df3 = pd.get_dummies(df3, columns=categorical_cols_to_encode, drop_first=True)

# Identifies target column from the features
target_column = 'price'
features = [col for col in df3.columns if col != target_column]

x = df3[features].copy()
y = df3[target_column].copy()

# Splits the data 80/20 for training and testing the model respectfully
x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42)

print("=" * 60)
print("DATASET 3: Dataset 1.csv")
print("=" * 60)
print("\nFirst 10 rows:")
display(df3.head(10))
print("\nData types:")
print(df3.dtypes)
print("\nMissing values:")
print(df3.isnull().sum())
print("\nTarget variable (price) statistics:")
print(df3['price'].describe())

### Linear Regression for Dataset 3

In [ ]:
from sklearn.model_selection import cross_val_score, KFold,cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score,root_mean_squared_error,get_scorer_names
import numpy as np
import matplotlib.pyplot as plt
from sklearn import metrics

model = LinearRegression()
model.fit(x_train,y_train)
predictions = model.predict(x_test)

print("Before cross validation")
print(f"RMSE: ${metrics.root_mean_squared_error(y_test, predictions):,.2f}")
print(f"MAE: ${metrics.mean_absolute_error(y_test, predictions):,.2f}")
print(f"r2: {metrics.r2_score(y_test, predictions):.4f}")

cv_results = cross_validate(model,x_train,y_train,cv=5,scoring=('neg_mean_absolute_error','neg_root_mean_squared_error','r2'),return_train_score=True)
print("After cross validation")
print(f"RMSE: ${-cv_results['test_neg_root_mean_squared_error'].mean():,.2f}")
print(f"MAE: ${-cv_results['test_neg_mean_absolute_error'].mean():,.2f}")
print(f"r2: {cv_results['test_r2'].mean():.4f}")


plt.figure(figsize=(8, 6))
plt.scatter(y_test, predictions, alpha=0.4, s=15)
max_val = max(y_test.max(), predictions.max())
plt.plot([0, max_val], [0, max_val], 'r--', label='Perfect prediction')

plt.xlabel('Actual Price ($)')
plt.ylabel('Predicted Price ($)')
plt.title('Baseline Linear Regression: Predicted vs Actual')
plt.legend()
plt.tight_layout()
plt.show()

before cross validaion ,from our results, the mean square error was very large approximately 46 billion, which when working with prices of hunderds of thoursand which are then squared can happen making this less interpretable as it represents dollars.

However the root mean square error, sq_root(mse), explains that our model was off by $215 172 on averge when prdicting houses but when squaring the values they are susceptible to outliars, which the dataset has, visible in the scatter plot.

the r_sq value of 0.69 showed that our model predicts 69% of the variance of the dataset

 after cross_validation, we can see a decrese in results as the mean square error was reduced t0 approximately 40 billion, the root mean square error, was reduced to approximately 200 000, meaning there was a decrease in how much our prediction were off by but the r2 value was till approximately 0.69, meaning our model still represents 69% of the variance of the dataset.

from these results we can conclude the model predicted fairly .

limitation : the inclusion of outliars